# Занятие 3: старт EDA
Это заготовка загрузки и проверок, не готовый анализ. Используйте свои результаты второго этапа. Все следующие сравнения с оттоком выполняйте на периоде обучения; валидацию и тест не используйте для отбора гипотез.


In [ ]:
from pathlib import Path
import csv
from datetime import datetime, timedelta, timezone
import pandas as pd
import matplotlib.pyplot as plt

PREPARED = Path("../../lesson2_prepared")  # Измените на свою папку результатов занятия 2.
NEW_DATA = Path("../data")              # Для запуска из lesson3/starter.
print("Подготовленные таблицы:", PREPARED.resolve())
print("Новые данные:", NEW_DATA.resolve())
TZ = timezone(timedelta(hours=3))

def load_csv(path):
    return pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)

def time_moscow(text):
    value = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if value.tzinfo is None:
        value = value.replace(tzinfo=TZ)
    return value.astimezone(TZ)


In [ ]:
subscribers = load_csv(PREPARED / "subscribers_clean.csv")
events = load_csv(PREPARED / "contract_events_clean.csv")
monthly = load_csv(PREPARED / "customer_month.csv")
network = load_csv(NEW_DATA / "network_quality_monthly.csv")
context = load_csv(NEW_DATA / "business_context.csv")
for name, table in [("абоненты", subscribers), ("события", events), ("месячные строки", monthly), ("сеть", network), ("контекст", context)]:
    print(name, table.shape)


In [ ]:
assert not subscribers.duplicated(["customer_id"]).any()
assert not events.duplicated(["event_id"]).any()
assert not monthly.duplicated(["customer_id", "billing_month"]).any()
assert set(monthly.customer_id) <= set(subscribers.customer_id)
assert not network.duplicated(["region", "measurement_month"]).any()
assert set(network.region) <= set(subscribers.region)
assert not context.duplicated(["context_id"]).any()

network["probe_count"] = pd.to_numeric(network["probe_count"], errors="raise")
network["failed_probe_count"] = pd.to_numeric(network["failed_probe_count"], errors="raise")
assert (network.probe_count > 0).all()
assert network.failed_probe_count.between(0, network.probe_count).all()
network["failed_probe_rate"] = network.failed_probe_count / network.probe_count

# Не заменяйте неизвестный трафик нулём.
monthly["traffic_gb"] = pd.to_numeric(monthly["traffic_gb"].replace("", float("nan")), errors="raise")
print("Доля неизвестного трафика в исходной месячной таблице:", monthly.traffic_gb.isna().mean())
print("Это проверка входа, а не статистика исследовательской базы на T.")


## Дальше работает команда
1. Заполните план минимум четырёх гипотез.
2. Постройте `eda_sample`: активная база на T, доступный прошлый месяц, отдельно будущая метка. Запишите период, горизонт и правила границ.
3. Проверьте число строк и время доступности независимо от построения снимков.
4. Для EDA используйте согласованные даты обучения. Рассчитайте таблицы и шесть рисунков задания.
5. Для сетевых отчётов сравните `report_available_at` с T; неизвестное значение оставляйте пропуском.
6. Добавьте выводы и ограничения. Проверьте воспроизводимость запуском сверху вниз.

Если второй этап не завершён, исправьте входные таблицы до сравнения с оттоком.


In [ ]:
# Ниже добавьте собственный код исследования и итоговые проверки.
# Не запускайте обучение модели на этом этапе.
